In [ ]:
# ==========================================
# DataInterview Prep MVP
# Запуск в Google Colab одним блоком
# ==========================================

!pip -q install gradio requests pandas

import gradio as gr
import requests
import pandas as pd
import random
from datetime import datetime

# ==========================================
# БАЗА ВОПРОСОВ
# ==========================================

QUESTIONS = [
    {
        "topic": "SQL",
        "question": "Чем отличается ROW_NUMBER() от RANK()?"
    },
    {
        "topic": "SQL",
        "question": "Что такое оконные функции?"
    },
    {
        "topic": "PySpark",
        "question": "Для чего используется repartition()?"
    },
    {
        "topic": "PySpark",
        "question": "Чем отличается cache() от persist()?"
    },
    {
        "topic": "Airflow",
        "question": "Что такое DAG?"
    },
    {
        "topic": "Data Modeling",
        "question": "Что такое Star Schema?"
    },
    {
        "topic": "ML",
        "question": "Что такое переобучение модели?"
    },
    {
        "topic": "MLOps",
        "question": "Зачем нужен MLflow?"
    }
]

# ==========================================
# AI МЕНТОР (заглушка)
# ==========================================

def evaluate_answer(question, answer):
    if len(answer.strip()) < 20:
        return """
### ❌ Ответ слишком короткий

Попробуйте раскрыть мысль подробнее.

**Подсказка:**
Объясните концепцию своими словами и приведите пример.
"""

    score = random.randint(65, 95)

    hints = [
        "Добавьте пример из реального проекта.",
        "Расскажите про ограничения подхода.",
        "Упомяните производительность решения.",
        "Сравните с альтернативами.",
        "Добавьте детали реализации."
    ]

    followups = [
        "Почему вы выбрали именно этот подход?",
        "Какие edge cases здесь существуют?",
        "Как решение будет работать на больших данных?",
        "Как это реализуется в продакшене?",
        "Что может пойти не так?"
    ]

    feedback = f"""
### 🎯 Оценка: {score}/100

**Что хорошо:**
✅ Ответ связан с вопросом
✅ Используются технические термины
✅ Видно понимание темы

**Что можно улучшить:**
⚠ {random.choice(hints)}
⚠ Добавьте больше практических деталей

**Вопрос от Senior Interviewer:**
🤔 {random.choice(followups)}
"""

    return feedback

# ==========================================
# НОВЫЙ ВОПРОС
# ==========================================

def get_random_question():
    q = random.choice(QUESTIONS)
    return f"[{q['topic']}] {q['question']}"

# ==========================================
# HH API
# ==========================================

def load_jobs(keyword):
    url = "https://api.hh.ru/vacancies"

    params = {
        "text": keyword,
        "per_page": 20
    }

    try:
        r = requests.get(url, params=params)

        if r.status_code != 200:
            return pd.DataFrame([{"Ошибка": "Не удалось получить вакансии"}])

        items = r.json()["items"]
        result = []

        for item in items:
            salary = "Не указана"
            if item["salary"]:
                salary = str(item["salary"])

            result.append({
                "Вакансия": item["name"],
                "Компания": item["employer"]["name"],
                "Зарплата": salary,
                "Ссылка": item["alternate_url"]
            })

        return pd.DataFrame(result)
    except Exception as e:
        return pd.DataFrame([{"Ошибка": f"Ошибка загрузки: {str(e)}"}])

# ==========================================
# АНАЛИТИКА НАВЫКОВ
# ==========================================

def market_skills():
    return pd.DataFrame({
        "Навык": [
            "SQL", "Python", "Spark", "Airflow",
            "Kafka", "Docker", "Kubernetes", "ClickHouse"
        ],
        "Популярность": [98, 95, 82, 79, 75, 70, 65, 61]
    })

# ==========================================
# UI
# ==========================================

with gr.Blocks(theme=gr.themes.Soft(), title="DataInterview Prep") as demo:

    gr.Markdown("""
# 🚀 DataInterview Prep

### Подготовка к собеседованиям по Data Engineering и ML

---

✅ 2000+ вопросов
✅ 50+ компаний
✅ AI-наставник
✅ Вакансии с HH

---
""")

    with gr.Tab("🎯 Практика интервью"):
        question_box = gr.Textbox(
            value=get_random_question(),
            label="Вопрос",
            lines=4
        )

        refresh_btn = gr.Button("🔄 Новый вопрос")
        answer_box = gr.Textbox(
            label="Ваш ответ",
            lines=10,
            placeholder="Введите ответ..."
        )
        check_btn = gr.Button("🤖 Проверить через AI")
        feedback = gr.Markdown()

        refresh_btn.click(get_random_question, outputs=question_box)
        check_btn.click(evaluate_answer, inputs=[question_box, answer_box], outputs=feedback)

    with gr.Tab("💼 Вакансии HH"):
        role = gr.Dropdown(
            ["Data Engineer", "ML Engineer", "Data Scientist", "MLOps Engineer", "Data Analyst"],
            value="Data Engineer",
            label="Выберите направление"
        )
        load_btn = gr.Button("Загрузить вакансии")
        jobs = gr.Dataframe()
        load_btn.click(load_jobs, inputs=role, outputs=jobs)

    with gr.Tab("📈 Навыки рынка"):
        gr.Markdown("""
## Что чаще всего требуют работодатели
""")
        skills = gr.Dataframe(value=market_skills())

    with gr.Tab("📚 База знаний"):
        gr.Markdown("""
### Data Engineering Roadmap

1. SQL
2. Python
3. Pandas
4. PySpark
5. Airflow
6. Kafka
7. Docker
8. Kubernetes
9. Data Modeling
10. MLOps

---

### Часто спрашивают:

• Window Functions
• Spark Shuffle
• Slowly Changing Dimensions
• ETL vs ELT
• Kafka Partitions
• Airflow DAGs
• Docker Networking
""")

    with gr.Tab("👤 О проекте"):
        gr.Markdown("""
### DataInterview Prep

Платформа подготовки к интервью по:

* Data Engineering
* Machine Learning
* Data Science
* MLOps

Создана как AI-наставник для подготовки к собеседованиям.
""")

# Запуск приложения
demo.launch(share=True, debug=True)

/tmp/ipykernel_1923/966075504.py:166: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="DataInterview Prep") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f87fac40ad3cdae9ea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
